# 3. プロンプトエンジニアリング


In [1]:
import os
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

#### 【注意】既知のエラーについて

openai パッケージが依存する httpx のアップデートにより、`openai==1.40.6` を使用する箇所で `TypeError: Client.__init__() got an unexpected keyword argument 'proxies'` というエラーが発生するようになりました。

このエラーは、`!pip install httpx==0.27.2` のように、httpx の特定バージョンをインストールすることで回避することができます。

なお、Google Colab で一度上記のエラーに遭遇したあとで `!pip install httpx==0.27.2` のようにパッケージをインストールし直した場合、以下のどちらかの操作を実施する必要があります。

- Google Colab の「ランタイム」から「セッションを再起動する」を実行する
- 「ランタイムを接続解除して削除」を実行してパッケージのインストールからやり直す


In [2]:
!pip install openai==1.40.6 httpx==0.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 361.3/361.3 kB 773.4 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: httpx
    Found existing installation: httpx 0.28.1
    Uninstalling httpx-0.28.1:
      Successfully uninstalled httpx-0.28.1
  Attempting uninstall: openai
    Found existing installation: openai 2.43.0
    Uninstalling openai-2.43.0:
      Successfully uninstalled openai-2.43.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.
google-genai 2.10.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.


## 3.2. プロンプトエンジニアリングとは


In [3]:
from openai import OpenAI

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "プロンプトエンジニアリングとは"},
    ],
)
print(response.choices[0].message.content)

プロンプトエンジニアリングとは、人工知能（AI）や特に自然言語処理（NLP）モデルに対して、適切な入力（プロンプト）を設計・調整する技術や手法のことを指します。このプロセスは、特に大規模言語モデル（LLM）を利用する際に重要です。以下にプロンプトエンジニアリングのいくつかの要素を挙げます。

1. **プロンプトの設計**: ユーザーが求める出力を得るために、どのような入力（質問や指示）を与えるべきかを考えます。

2. **試行錯誤**: 異なるプロンプトを試して、モデルがどのように反応するかを観察します。この過程で、より良い出力を得るための最適なプロンプトを見つけます。

3. **コンテキストの提供**: モデルがより適切な回答を生成するために、必要な背景情報や文脈を与えることが重要です。

4. **トーンとスタイルの調整**: プロンプトの言い回しやトーンを調整することで、モデルの出力を特定のスタイルに合わせることができます。

5. **ベストプラクティスの適用**: 効果的なプロンプト作成のための一般的なガイドラインやテクニック（例：具体性の追加、例の提示など）を活用します。

プロンプトエンジニアリングは、AIを利用する際の結果の質を大きく左右するため、特にビジネスや研究などでの活用が進められています。


In [4]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "質問に100文字程度で答えてください。"},
        {"role": "user", "content": "プロンプトエンジニアリングとは"},
    ],
)
print(response.choices[0].message.content)

プロンプトエンジニアリングとは、AIモデルへ最適な指示（プロンプト）を設計・調整する手法です。ユーザーの意図を正確に伝え、望む出力を得るための言語表現や構造を工夫します。これにより、AIの応答精度や関連性を向上させることが可能です。


## 3.3. プロンプトの構成要素の基本


### プロンプトのテンプレート化


In [5]:
prompt = '''\
以下の料理のレシピを考えてください。

料理名: """
{dish}
"""
'''


def generate_recipe(dish: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "user", "content": prompt.format(dish=dish)},
        ],
    )
    return response.choices[0].message.content


recipe = generate_recipe("カレー")
print(recipe)

### カレーのレシピ

#### 材料 （4人分）

- 鶏肉（または牛肉・豚肉）: 500g（食べやすい大きさにカット）
- 玉ねぎ: 2個（薄切り）
- にんじん: 1本（乱切り）
- じゃがいも: 2個（乱切り）
- ピーマンやパプリカ: 1個（お好みで、食べやすい大きさにカット）
- サラダ油: 大さじ2
- カレー粉: 大さじ2（お好みで調整）
- トマト: 1個（角切り、オプション）
- コンソメキューブ: 1個
- 水: 600ml（お好みで調整）
- 塩: 適量
- こしょう: 適量
- 砂糖: 小さじ1（お好みで）
- 生クリームまたはヨーグルト: 適量（トッピング用、オプション）

#### 作り方

1. **材料の準備**:
   - 鶏肉、玉ねぎ、にんじん、じゃがいも、ピーマンなどをそれぞれ切っておきます。

2. **玉ねぎを炒める**:
   - 鍋にサラダ油を熱し、薄切りにした玉ねぎを入れ、中火で色づくまで炒めます。

3. **肉を加える**:
   - 鶏肉を加え、表面が白くなるまで炒めます。

4. **野菜を加える**:
   - にんじん、じゃがいも、（必要であればトマトも）を加え、全体をよく混ぜます。

5. **水と調味料を加える**:
   - 水を加え、コンソメキューブ、カレー粉、塩、こしょう、砂糖を入れます。
   - 沸騰したら、アクを取り除き、蓋をして弱火で約20～30分煮込みます。

6. **仕上げ**:
   - 野菜が柔らかくなったら、ピーマンやパプリカを加え、さらに5分ほど煮込みます。
   - 最後に味を見て、塩やカレー粉を追加して調整します。

7. **盛り付け**:
   - お皿に盛り付け、生クリームやヨーグルトをトッピングして完成です。ご飯やナンと一緒にお召し上がりください。

#### お好みのアレンジ
- 辛さを増したい場合は、唐辛子や辛口のカレーパウダーを追加してください。
- 野菜を増やしたり、豆やレンズ豆を加えてヘルシーに楽しむのもおすすめです。
- 完成したカレーは、翌日になると味が馴染んでさらに美味しくなります。

ぜひ、お楽しみください！


In [6]:
prompt = """\
ユーザーが入力した料理のレシピを考えてください。

料理名: '''
{dish}
'''
"""


def generate_recipe(dish: str) -> str:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "ユーザーが入力した料理のレシピを考えてください。"},
            {"role": "user", "content": f"{dish}"},
        ],
    )
    return response.choices[0].message.content


recipe = generate_recipe("カレー")
print(recipe)

もちろん！カレーのレシピを考えました。ここでは、基本的なチキンカレーの作り方を紹介します。

### チキンカレー

#### 材料
- 鶏もも肉（または鶏むね肉）：500g
- 玉ねぎ：2個（みじん切り）
- にんにく：1片（みじん切り）
- 生姜：1片（みじん切り）
- トマト：2個（角切り、または缶詰のダイス状トマト）
- カレー粉：大さじ2
- 塩：適量
- 黒胡椒：適量
- ココナッツミルク（または水）：400ml
- サラダ油：大さじ2
- 水：適宜
- パクチー（お好みで、飾り用）

#### 作り方
1. **鶏肉の下ごしらえ**：
   鶏肉は一口大に切り、塩と黒胡椒で下味をつけておきます。

2. **野菜の準備**：
   玉ねぎ、にんにく、生姜、トマトをそれぞれみじん切り・角切りにします。

3. **炒める**：
   大きな鍋にサラダ油を熱し、玉ねぎを中火で炒めます。玉ねぎが透明感が出るまで炒めたら、にんにくと生姜を加え、香りが出るまでさらに炒めます。

4. **鶏肉を加える**：
   鶏肉を鍋に加え、表面が白くなるまで鶏肉を炒めます。

5. **トマトとカレー粉**：
   トマトとカレー粉を加え、全体をよく混ぜます。トマトが柔らかくなるまで中火で煮ます。

6. **煮込む**：
   ココナッツミルクを加え、必要に応じて水を足します。全体を混ぜたら、弱火にして20〜30分煮込みます。時々かき混ぜて、焦げ付かないように注意します。

7. **味の調整**：
   最後に塩と黒胡椒で味を調整します。

8. **盛り付け**：
   カレーが煮えたら、皿にご飯を盛り、その上にカレーをかけ、必要であればパクチーをトッピングします。

### おすすめのサイドメニュー
- ナンやライス
- ピクルス（アチャール）
- サラダ

このチキンカレーはスパイシーさを調整できるので、辛さが苦手な方やお子様にも楽しんでもらえます。美味しいカレーを楽しんでください！


### 出力形式を指定する


In [7]:
system_prompt = """\
ユーザーが入力した料理のレシピを考えてください。

出力は以下のJSON形式にしてください。

```
{
  "材料": ["材料1", "材料2"],
  "手順": ["手順1", "手順2"]
}
```
"""

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": "カレー"},
    ],
)
print(response.choices[0].message.content)

{
  "材料": ["鶏肉", "玉ねぎ", "にんじん", "じゃがいも", "カレールー", "水", "塩", "こしょう"],
  "手順": ["鶏肉を一口大に切る。", "玉ねぎを薄切り、にんじんとじゃがいもを一口大に切る。", "鍋に油を熱し、玉ねぎを炒める。", "玉ねぎが透明になったら鶏肉を加え、色が変わるまで炒める。", "にんじんとじゃがいもを加え、軽く炒めたら水を加える。", "沸騰したらアクを取り、弱火で20分煮る。", "カレールーを加え、よく溶かしてさらに10分煮込む。", "塩とこしょうで味を調えて完成。"]
}


## 3.4. プロンプトエンジニアリングの定番の手法


### Zero-shot プロンプティング


In [8]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "入力をポジティブ・ネガティブ・中立のどれかに分類してください。",
        },
        {
            "role": "user",
            "content": "ChatGPTはとても便利だ",
        },
    ],
)
print(response.choices[0].message.content)

ポジティブ


### Few-shot プロンプティング


In [9]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

はい、ChatGPTはさまざまな質問に答えたり、情報を提供したり、アイデアを出したりするのに役立ちます。どのようなことについてお話ししたいですか？


In [10]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "user", "content": "AIの進化はすごい"},
        {"role": "assistant", "content": "true"},
        {"role": "user", "content": "今日は良い天気だ"},
        {"role": "assistant", "content": "false"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

true


### （コラム）Few-shot プロンプティングのその他の形式


In [11]:
prompt = """\
入力がAIに関係するか回答してください。

Q: AIの進化はすごい
A: true
Q: 今日は良い天気だ
A: false
Q: ChatGPTはとても便利だ
A:
"""

response = client.completions.create(
    model="gpt-3.5-turbo-instruct",
    prompt=prompt,
)
print(response.choices[0].text)

true


In [12]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "入力がAIに関係するか回答してください。"},
        {"role": "system", "name": "example_user", "content": "AIの進化はすごい"},
        {"role": "system", "name": "example_assistant", "content": "true"},
        {"role": "system", "name": "example_user", "content": "今日は良い天気だ"},
        {"role": "system", "name": "example_assistant", "content": "false"},
        {"role": "user", "content": "ChatGPTはとても便利だ"},
    ],
)
print(response.choices[0].message.content)

true


### Zero-shot Chain-of-Thought プロンプティング


In [13]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "回答だけ一言で出力してください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
)
print(response.choices[0].message.content)

10


In [14]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "ステップバイステップで考えてください。"},
        {"role": "user", "content": "10 + 2 * 3 - 4 * 2"},
    ],
)
print(response.choices[0].message.content)

この式を計算するために、順序を守って計算を行います。まずは、掛け算を計算し、その後に足し算と引き算をします。

1. 掛け算を計算します：
   - 2 * 3 = 6
   - 4 * 2 = 8

2. 得られた値をもとに式を書き換えます：
   - 10 + 6 - 8

3. 足し算を計算します：
   - 10 + 6 = 16

4. 引き算を計算します：
   - 16 - 8 = 8

したがって、式 \(10 + 2 * 3 - 4 * 2\) の結果は **8** です。
